# Files, Folders & Permissions

## Where we're going

You already know how to *find* files (with `cd` and `ls`). Now you'll learn how to *manipulate* them — create, view, copy, move, delete, link — and how to control *who* can do what to them through permissions.

The roadmap for this notebook:

1. **Creating** — get new files and directories into existence.
2. **Viewing** — read what's in them, big or small.
3. **Moving things around** — copy, move/rename, delete.
4. **Linking** — two names for the same file, or a pointer to another file.
5. **File types** — the seven kinds of things you'll see with `ls -l`.
6. **Permissions** — the model, then `chmod`, `chown`, and `umask`.
7. **Hidden files** — the dotfile convention.

Linux's file model is shared by every major Linux distribution and matches what you'll be tested on in the LFCS exam. It's also where new Linux users make the most mistakes — particularly around permissions and `rm`. Take your time with this notebook.

## Creating files and directories — `touch` and `mkdir`

To create an **empty file**, use **`touch`**:

```bash
touch report.txt
```

`touch` was originally designed to update a file's timestamps (and it still does — touching an existing file updates its modification time). But its side effect of creating empty files when none exist is what you'll use it for most.

To create a **directory**, use **`mkdir`**:

```bash
mkdir projects
```

To create a chain of nested directories in one step, use **`mkdir -p`** ("p" for "parents"):

```bash
mkdir -p projects/linux/notes
```

Without `-p`, `mkdir` errors out if any intermediate directory doesn't exist. With `-p`, it creates each missing parent. It also doesn't complain if the directory already exists, which makes it safe to use in scripts.

A couple of practical points:

- **`touch` accepts multiple arguments**: `touch a.txt b.txt c.txt` creates three files in one call.
- **`mkdir` accepts multiple arguments** too: `mkdir dir1 dir2 dir3`.
- **Linux filenames are case-sensitive.** `Report.txt` and `report.txt` are different files. This catches Windows and macOS users off-guard.
- **Spaces in filenames** work, but you must quote them: `touch "my notes.txt"`. The convention on Linux is to avoid spaces — use dashes or underscores instead (`my-notes.txt`, `my_notes.txt`).

In [ ]:
!cd /tmp && touch report.txt notes.txt && ls -l report.txt notes.txt
!cd /tmp && mkdir -p projects/linux/notes && ls -R projects
!cd /tmp && rm -rf projects report.txt notes.txt

## Viewing file contents — `cat`, `less`, `head`, `tail`, `more`

Five tools for reading a file. Picking the right one matters:

**`cat <file>`** — print the whole file to your terminal. Fine for small files (a few dozen lines). Don't use it on a 50,000-line log file or you'll regret it. The name is short for "concatenate" — `cat file1 file2` prints both, glued together.

**`less <file>`** — open the file in an interactive pager. Use this for anything large. Inside `less`:

- **Space** or **PgDn** — next page
- **b** or **PgUp** — previous page
- **/pattern** — search forwards
- **n** — next search match, **N** — previous
- **g** — go to start, **G** — go to end
- **q** — quit

`less` is **fast** — it doesn't load the whole file into memory, just the part you're looking at. You can `less` a 10 GB log file and it opens instantly.

**`head <file>`** — print the first 10 lines. Override with `-n N`: `head -n 5 file.log` shows the first 5 lines.

**`tail <file>`** — print the last 10 lines. Same `-n` flag. The killer feature is `tail -f`: **"follow"** mode that keeps printing new lines as they're appended. The standard way to watch a log file: `tail -f /var/log/syslog`.

**`more`** is an older, simpler pager. `less` is "more than more" (a deliberate pun). Use `less`.

**`file <path>`** — not for *reading* a file, but for **identifying** what it is. Linux doesn't trust file extensions; `file` reads the first few bytes (the "magic number") to tell you what's actually inside. `file /bin/ls` says "ELF 64-bit LSB executable"; `file /etc/passwd` says "ASCII text"; `file image.png` says "PNG image data".

In [ ]:
!cat /etc/hostname
!head -5 /etc/passwd
!tail -3 /etc/passwd
!wc -l /etc/passwd
!file /etc/passwd /bin/ls /etc/hostname

## Copying — `cp`

To copy a file: `cp <source> <destination>`.

```bash
cp report.txt report-backup.txt           # copy in the same directory
cp report.txt /tmp/                       # copy into a directory (keeps the name)
cp report.txt /tmp/new-name.txt           # copy and rename
cp file1 file2 file3 /tmp/                # copy multiple files into a directory
```

To copy a **directory** and its contents, you need **`-r`** ("recursive"):

```bash
cp -r projects/ /backup/
```

Without `-r`, copying a directory fails with "omitting directory" — `cp` refuses by default to protect you from common mistakes.

Useful flags:

- **`-i`** ("interactive") — prompt before overwriting an existing file. Worth aliasing `cp` to `cp -i` for safety.
- **`-v`** ("verbose") — print every file as it's copied. Useful for big copies so you know it's making progress.
- **`-p`** ("preserve") — keep the original's timestamps, ownership, and permissions instead of resetting them.
- **`-n`** ("no-clobber") — never overwrite. Skip files that already exist at the destination.
- **`-u`** ("update") — only copy if the source is newer than the destination, or the destination doesn't exist. Useful for incremental backups.

**Quick warning:** `cp` overwrites destinations silently by default. If `report-backup.txt` already exists and you `cp report.txt report-backup.txt`, the old backup is gone with no warning. That's why `-i` is a useful habit.

For copying *between* machines, you'll want **`scp`** or **`rsync`** — covered in notebook 08.

In [ ]:
!cd /tmp && echo "version 1" > original.txt && cp -v original.txt copy.txt && cat copy.txt
!cd /tmp && mkdir -p src/sub && echo "file" > src/a.txt && echo "nested" > src/sub/b.txt
!cd /tmp && cp -rv src dest && ls -R dest
!cd /tmp && rm -rf original.txt copy.txt src dest

## Moving and renaming — `mv`

In Linux, **renaming is the same operation as moving** — there's no separate `rename` command in the basics. Both are done with **`mv`**.

```bash
mv report.txt report-final.txt           # rename in the same directory
mv report.txt /tmp/                      # move into another directory
mv report.txt /tmp/renamed.txt           # move AND rename
mv file1 file2 file3 /tmp/               # move multiple files into a directory
```

When source and destination are on the **same filesystem**, `mv` is nearly instant — it just rewrites a directory entry. When they're on different filesystems (e.g. from your home directory to a USB stick), `mv` falls back to copy-then-delete, which can take much longer for large files.

Useful flags (same as `cp`):

- **`-i`** — prompt before overwriting.
- **`-v`** — verbose.
- **`-n`** — never overwrite.
- **`-u`** — only move if source is newer.

There's **no `-r` flag for `mv`** — moving a directory always includes its contents, no questions asked. `mv old-dir new-name` just works.

If you genuinely want to *rename* without moving, you can stay in the same directory: `mv old.txt new.txt`. Some distros also ship a `rename` command (Debian/Ubuntu's is Perl-based and supports regex; the RHEL-family `rename` is simpler) — useful for bulk renames, but the syntaxes differ, so use with care.

In [ ]:
!cd /tmp && touch original.txt && mv -v original.txt renamed.txt && ls original.txt 2>/dev/null; ls renamed.txt
!cd /tmp && mkdir -p movetest && mv -v renamed.txt movetest/ && ls movetest
!cd /tmp && rm -rf movetest renamed.txt

## Deleting — `rm` and the careful use of `rm -r`

**`rm <file>`** deletes a file. No trash can. No undo. Gone.

```bash
rm report.txt
rm file1 file2 file3
```

To delete a **directory and everything inside it**, you need **`-r`** ("recursive"):

```bash
rm -r old-project/
```

To suppress prompts and force-delete read-only files: **`-f`** ("force").

```bash
rm -rf temp-stuff/
```

`rm -rf` is genuinely dangerous. The "joke" `rm -rf /` (delete everything starting from the root) has ended real careers and taken down real production servers. Modern `rm` refuses `rm -rf /` by default (via the `--preserve-root` option, which is on by default), but `rm -rf /home/somebody-important/` still works without complaint, and `rm -rf /` with `--no-preserve-root` will dutifully try.

Habits that save you from `rm` accidents:

- **Use `rm -i`** for interactive confirmation, especially when wildcards are involved. `rm -i *.log` asks before each delete.
- **List first, then delete.** If you're going to `rm *.tmp`, run `ls *.tmp` first to see exactly what would be matched. If the list looks right, recall the command (up arrow) and change `ls` to `rm`.
- **Be extra careful with variables in deletion commands.** `rm -rf "$dir/"` where `$dir` happens to be empty becomes `rm -rf /` — exactly the disaster you don't want. Always double-check variables before deletion.
- **For directories**, `rmdir <dir>` (no `-r`) deletes only **empty** directories — a safer choice when you mean "this directory should be empty, remove it" rather than "delete this and everything in it".

Linux distributions don't have a built-in trash like macOS or Windows. If you delete with `rm`, only specialised recovery tools have any chance, and only sometimes.

In [ ]:
!cd /tmp && touch to-delete.txt && rm -v to-delete.txt
!cd /tmp && mkdir empty-dir && rmdir empty-dir
!cd /tmp && mkdir -p deltest/sub && touch deltest/a deltest/sub/b && rm -rv deltest

## Links — hard links and symbolic links

A **link** lets one file have multiple names — or lets a file act as a pointer to another. Linux supports two kinds.

**Hard link** — created with `ln <source> <link-name>`. A hard link is a second directory entry that points at the **same underlying data**. Both names are equally first-class — there's no "original" and "copy". If you change the content through one name, the other sees the change. If you delete one name, the data stays as long as at least one name still points to it.

```bash
ln original.txt second-name.txt          # both names now point to the same content
```

**Symbolic link (symlink)** — created with `ln -s <target> <link-name>`. A symlink is a tiny separate file whose content is *the path to another file*. When you read the symlink, the kernel transparently follows it to the target.

```bash
ln -s /etc/hostname my-hostname-link     # my-hostname-link points to /etc/hostname
```

The differences that matter for now:

| | Hard link | Symbolic link |
|---|---|---|
| What it is | second name for the same data | a pointer to another path |
| Survives target rename? | yes (both names are equal) | no (the path becomes invalid) |
| Can target a directory? | no (kernel forbids it) | yes |
| Can cross filesystems (different disks)? | no | yes |
| Shows up in `ls -l` as | a regular file (`-`) | a link (`l`) with `→ target` |

**Symbolic links are what you'll use 99% of the time.** They appear constantly in Linux:

- `/lib` is usually a symlink to `/usr/lib`.
- `/bin/sh` is a symlink to `/bin/bash` (or `/bin/dash` on Debian/Ubuntu).
- Package managers create `/usr/bin/python3` as a symlink to a specific Python version.
- The `~/.bashrc` you customise is often a symlink to a file in a dotfiles repo.

You can spot a symlink in `ls -l` by the `l` type character and the `->` showing what it points to. **`readlink -f <symlink>`** resolves a symlink chain all the way to the final target.

In [ ]:
!cd /tmp && echo "shared content" > target.txt && ln target.txt hardlink.txt && ln -s target.txt symlink.txt
!ls -l /tmp/target.txt /tmp/hardlink.txt /tmp/symlink.txt
!cat /tmp/symlink.txt
!readlink /tmp/symlink.txt
!cd /tmp && rm -f target.txt hardlink.txt symlink.txt

## The seven file types

When you run `ls -l`, the first character of each line tells you what kind of thing you're looking at. Linux recognises seven types:

| Character | Type | What it is |
|---|---|---|
| **`-`** | regular file | text, binaries, images, documents — most files |
| **`d`** | directory | a folder; holds other files |
| **`l`** | symbolic link | a pointer to another file |
| **`c`** | character device | unbuffered byte stream — keyboards, terminals, `/dev/null`, `/dev/random` |
| **`b`** | block device | fixed-size random-access blocks — disks like `/dev/sda`, `/dev/nvme0n1` |
| **`s`** | socket | a communication endpoint between processes — `/var/run/docker.sock` |
| **`p`** | named pipe (FIFO) | a persistent pipe on the filesystem |

You'll mostly deal with the first three. The other four are for the curious:

- **Character and block devices** in `/dev` are how userspace talks to hardware (the kernel handles the real work). Reading from `/dev/random` gives you random bytes; writing to `/dev/null` discards everything.
- **Sockets** are how local services communicate. Database servers, Docker, and system services often listen on a socket file in `/var/run/` or `/tmp/`.
- **Named pipes** are rare in everyday use but show up in some scripted setups.

The `file` command (from earlier) tells you about regular files' contents (text, image, executable, etc.). The first character of `ls -l` tells you the **type** as the kernel sees it.

In [ ]:
!ls -l /dev/null /dev/sda /etc /etc/hostname /var/run/docker.sock 2>/dev/null

## The permissions model — who can do what

Linux is a multi-user system from the ground up. Every file has an **owner** (a user), a **group**, and a set of **permissions** that control what the owner, the group, and everyone else can do with it.

Permissions split into three **classes** and three **actions**:

**Three classes — who's asking?**

- **User (u)** — the file's owner.
- **Group (g)** — members of the file's group.
- **Other (o)** — everyone else who isn't the owner and isn't in the group.

**Three actions — what are they trying to do?**

For **files**:

- **Read (r)** — view the contents.
- **Write (w)** — modify or overwrite the contents.
- **Execute (x)** — run the file (if it's a program or script).

For **directories** (the meanings shift a bit):

- **Read (r)** — list the files inside (e.g. `ls`).
- **Write (w)** — create, delete, or rename files inside.
- **Execute (x)** — enter the directory (`cd` into it), and access files inside.

A directory **without execute** for you is effectively locked — even if you can read it, you can't `cd` into it and you can't access anything within. **Execute on a directory means "traversable."** This trips up new users frequently.

Combining the classes and actions gives nine permission bits per file:

```
       owner     group     other
        ↓         ↓         ↓
   [ r w x ] [ r w x ] [ r w x ]
```

You'll see these every time you run `ls -l`.

## Reading `ls -l` output, character by character

A line from `ls -l` looks like this:

```
-rw-r--r-- 1 ubuntu ubuntu 1234 Mar 14 10:23 report.txt
```

Read it left to right:

```
- rw- r-- r-- 1 ubuntu ubuntu 1234 Mar 14 10:23 report.txt
│ ─┬─ ─┬─ ─┬─ │ ──┬── ──┬── ──┬─ ─────┬────── ─────┬─────
│  │   │   │  │   │      │     │       │             │
│  │   │   │  │   │      │     │       │             └── filename
│  │   │   │  │   │      │     │       └── last modification time
│  │   │   │  │   │      │     └── size in bytes
│  │   │   │  │   │      └── group name
│  │   │   │  │   └── owner name
│  │   │   │  └── number of hard links to this file
│  │   │   └── permissions for other (everyone else): r-- = read only
│  │   └── permissions for group: r-- = read only
│  └── permissions for user (owner): rw- = read and write, no execute
└── file type: `-` = regular file
```

So that file is:

- A **regular file** (`-`)
- The owner (`ubuntu`) can **read and write**, but not execute
- Members of the group (`ubuntu`) can **read** only
- Everyone else can **read** only
- One hard link points to it (the file itself)
- It's 1234 bytes
- Last modified on March 14 at 10:23
- Named `report.txt`

A directory looks similar but starts with `d` and typically shows `rwxr-xr-x` — the owner can do everything, others can read and traverse but not modify:

```
drwxr-xr-x 5 ubuntu ubuntu 4096 Mar 14 10:23 projects/
```

If you see `-rwxr-xr-x`, that's an executable file (a program or script) — owner has full control, everyone can run it.

Spend a minute reading a few lines of `ls -l /etc/` until this clicks. It's the most important "ground truth" you'll read in Linux.

In [ ]:
!ls -l /etc/hostname /etc/passwd /bin/ls /etc /tmp

## `chmod` — changing permissions

`chmod` has two equivalent syntaxes. Pick whichever you prefer; both are standard. Learn to read both because you'll see both in the wild.

### Symbolic form — like a sentence

```
chmod [who][operator][what] file
```

- **who**: `u` (user/owner), `g` (group), `o` (other), `a` (all = u+g+o)
- **operator**: `+` (add), `-` (remove), `=` (set exactly to)
- **what**: `r`, `w`, `x` (any combination)

Examples:

```bash
chmod u+x script.sh          # add execute for owner
chmod go-w file.txt          # remove write for group and other
chmod a+r public.txt         # everyone can read
chmod o= secret.txt          # remove ALL permissions for other (= with nothing means none)
chmod u=rw,go=r file.txt     # set exactly: owner rw, group r, other r
```

### Numeric (octal) form — three digits

Each class (user, group, other) gets one digit. The digit is the sum of the permissions:

| Number | Permission |
|---|---|
| 4 | read |
| 2 | write |
| 1 | execute |
| 0 | none |

You add them together:

- `7` = 4+2+1 = read + write + execute (rwx)
- `6` = 4+2 = read + write (rw-)
- `5` = 4+1 = read + execute (r-x)
- `4` = read only (r--)
- `0` = nothing (---)

Then you put one digit per class in order: **user, group, other**.

The combinations you'll see constantly:

| Octal | Symbolic | Used for |
|---|---|---|
| **`755`** | `rwxr-xr-x` | executable programs, public directories — owner can do everything, others can read and run |
| **`644`** | `rw-r--r--` | regular text/config files — owner can edit, others can read |
| **`700`** | `rwx------` | private directories like `~/.ssh` — only owner can access |
| **`600`** | `rw-------` | private files like `~/.ssh/id_rsa` — only owner can read/write |
| **`777`** | `rwxrwxrwx` | "anyone can do anything" — almost always a security smell |

```bash
chmod 755 script.sh
chmod 644 notes.txt
chmod 700 ~/.ssh
chmod 600 ~/.ssh/id_rsa
```

Two useful flags:

- **`-R`** — recursive. `chmod -R 755 some-dir/` changes permissions on the directory and everything inside.
- **`-v`** — verbose. Prints what's being changed.

**A common gotcha:** `chmod -R 755 some-dir/` is usually wrong for a mix of files and subdirectories — it makes all your regular files executable. The safe pattern is to apply different perms to files and directories:

```bash
find some-dir -type d -exec chmod 755 {} +    # directories: 755
find some-dir -type f -exec chmod 644 {} +    # files: 644
```

We'll come back to `find` in notebook 04.

In [ ]:
!cd /tmp && touch test-perms.txt && ls -l test-perms.txt
!cd /tmp && chmod 600 test-perms.txt && ls -l test-perms.txt
!cd /tmp && chmod u+x test-perms.txt && ls -l test-perms.txt
!cd /tmp && chmod 644 test-perms.txt && ls -l test-perms.txt
!cd /tmp && rm test-perms.txt

## `chown` and `chgrp` — changing ownership

The owner and group of a file are stored alongside the permissions. You change them with `chown` (change owner) and `chgrp` (change group), or both at once with `chown owner:group`.

```bash
sudo chown alice file.txt              # change owner to alice
sudo chown alice:devs file.txt         # change owner to alice AND group to devs
sudo chown :devs file.txt              # change only the group (note the leading colon)
sudo chgrp devs file.txt               # same effect — change group only
sudo chown -R alice:devs /srv/app/     # recursive: change ownership of a whole tree
```

**You almost always need `sudo` for `chown`.** Regular users can't change file ownership to someone else (otherwise security would collapse — anyone could give themselves files they shouldn't have). Only `root` can.

`chgrp` is slightly more permissive — a regular user can change a file's group to another group **they belong to**, if they own the file. But in practice you'll see `sudo chgrp` most of the time too.

A common task: when you `sudo` to install something, the resulting files end up owned by `root`. To make them yours:

```bash
sudo chown -R $USER:$USER /opt/my-thing/
```

This recursively gives ownership back to your user account. Notebook 06 covers users and groups in detail; for now, just know that `chown` is how you reassign files between accounts.

## `umask` — what new files get by default

When you create a file with `touch`, the kernel doesn't ask you what permissions it should have. It applies a **default** — usually `rw-r--r--` (644) for files, `rwxr-xr-x` (755) for directories.

The default isn't fixed. It's calculated from your **umask** ("user file-creation mask"), a number that says **which permissions to *remove* from the maximum**.

The maximum permissions a new file is allowed to have:

- For **files**: `666` (rw-rw-rw-) — never executable by default.
- For **directories**: `777` (rwxrwxrwx).

The default `umask` is usually `022`. This is a mask of bits to **subtract**:

- File creation: `666` − `022` = `644` → `rw-r--r--`
- Directory creation: `777` − `022` = `755` → `rwxr-xr-x`

So `umask 022` means: "remove write permission from group and other." Hence the common 644/755 you see everywhere.

A more private default, common on servers, is `umask 077`:

- Files: `666` − `077` = `600` → `rw-------`
- Directories: `777` − `077` = `700` → `rwx------`

This makes new files completely private to the owner. Useful when you're working with sensitive data.

To see your current umask:

```bash
umask
```

To change it for the current shell:

```bash
umask 077
```

To make it permanent, set it in your shell's startup file (`~/.bashrc` or `~/.profile`). Notebook 06 covers shell startup files.

The number you set is the mask itself. Don't try to set the result — set what should be **removed** from `666`/`777`.

In [ ]:
!umask
!cd /tmp && umask 022 && touch default-022.txt && ls -l default-022.txt
!cd /tmp && umask 077 && touch default-077.txt && ls -l default-077.txt
!cd /tmp && rm -f default-022.txt default-077.txt

## Hidden files and dotfiles

A file or directory whose **name starts with a dot** is **hidden** — it doesn't show up in plain `ls` output. This isn't a security feature; it's a convention. `ls -a` shows them; `ls` doesn't.

```
.bashrc           ← hidden
.config/          ← hidden
.ssh/             ← hidden
normal.txt        ← not hidden
```

The convention exists because **almost all Linux configuration is per-user, lives in your home directory, and would clutter `ls` output otherwise**. Take a peek:

```bash
ls -la ~ | head -20
```

You'll see things like:

| File | What it's for |
|---|---|
| `.bashrc` | bash startup file — runs every time you open a new shell |
| `.bash_profile` or `.profile` | runs on login (a more specific variant) |
| `.bash_history` | record of every command you've typed |
| `.ssh/` | SSH keys and known hosts |
| `.config/` | per-application configuration (modern apps use this) |
| `.local/` | per-user installed binaries and data |
| `.cache/` | per-user caches |

These are universally called **dotfiles**. Many Linux users keep their dotfiles in a public git repository — search "dotfiles" on GitHub for thousands of examples. It's how experienced users sync their environment across machines.

Two practical points:

- **`ls -a`** shows hidden files. **`ls -la`** combines hidden with long format — the form you'll use most when investigating a home directory.
- **`.` and `..`** also appear in `ls -a` output. They're not hidden files — they're the **current directory** (`.`) and the **parent directory** (`..`), which is why `cd ..` works.

In [ ]:
!ls -la ~ | head -15
!ls -d ~/.* 2>/dev/null | head -10